# Expense Tracker — Kaggle Master Training

**Run All is enough for the standard training pipeline.** The notebook clones the ML code, installs dependencies, automatically downloads the configured Hugging Face datasets, caches normalized data, trains the models, evaluates them, and leaves all reports/graphs/models under `/kaggle/working`.

Standard datasets: `mitulshah/transaction-categorization`, `Ranjit0034/finee-dataset`, and `Sumeetgpt/indian-transaction-categorization-synthetic`.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO = Path('/kaggle/working/expense-tracker')
BRANCH = 'feature/ml-expense-intelligence'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yoge-2004/expense-tracker.git', str(REPO)], check=True)
ML = REPO / 'ml'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ML)], check=True)
print('ML code:', ML)

In [ ]:
import os, json
from expense_ml.resources import configure_resources

os.environ.setdefault('EXPENSE_ML_CPU_THREADS', 'auto')
os.environ.setdefault('EXPENSE_ML_DATALOADER_WORKERS', '8')
os.environ.setdefault('EXPENSE_ML_BATCH_SIZE', '32')
os.environ.setdefault('EXPENSE_ML_EVAL_BATCH_SIZE', '64')
os.environ.setdefault('EXPENSE_ML_MIXED_PRECISION', 'auto')
os.environ.setdefault('EXPENSE_ML_MAX_MERCHANTS', '250000')
os.environ.setdefault('EXPENSE_ML_DUPLICATE_MAX_ROWS', '500000')
os.environ.setdefault('EXPENSE_ML_NORMALIZE_CHUNK_SIZE', '250000')
os.environ['EXPENSE_ML_OUTPUT'] = '/kaggle/working/expense-ml-runs'
os.environ['EXPENSE_ML_DATA_CACHE'] = '/kaggle/working/expense-ml-data'
print(json.dumps(configure_resources(), indent=2))

In [ ]:
# Download the standard datasets from Hugging Face and normalize/cache them.
from expense_ml.data.fetch import fetch_configured_datasets
import yaml

config_path = ML / 'config' / 'datasets.yaml'
config = yaml.safe_load(config_path.read_text())
prepared, manifest = fetch_configured_datasets(
    config['datasets'],
    cache_dir=Path('/kaggle/working/expense-ml-data'),
    progress=True,
)
prepared.to_parquet('/kaggle/working/expense-ml-data/transactions.parquet', index=False)
Path('/kaggle/working/expense-ml-data/fetch_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('Prepared rows:', len(prepared))

In [ ]:
# Full master run: classification + auxiliary models + reports/figures.
%run /kaggle/working/expense-tracker/ml/kaggle_train.py

In [ ]:
from pathlib import Path
import json

runs = sorted(Path('/kaggle/working/expense-ml-runs').glob('*'))
latest = runs[-1]
manifest = json.loads((latest / 'manifest.json').read_text())
print('Latest run:', latest)
print(json.dumps(manifest['resources'], indent=2))
print(json.dumps(manifest['models'], indent=2))
print((latest / 'reports' / 'REPORT.md').read_text())